In [2]:
!pip uninstall -y flwr flwr-nightly flwr-client flwr-server

In [3]:
!pip uninstall -y flwr tensorflow ml-dtypes
!pip install flwr[simulation]==1.9.0

Found existing installation: tensorflow 2.19.0
Uninstalling tensorflow-2.19.0:
^C
ERROR: Operation cancelled by user
^C
ERROR: Operation cancelled by user


In [4]:
!pip uninstall -y tensorflow jax jaxlib ml-dtypes
!pip install --upgrade flwr[simulation]
!pip install numpy==2.0.1 ml-dtypes==0.5.0 protobuf==5.29.1

Found existing installation: jax 0.7.2
Uninstalling jax-0.7.2:
  Successfully uninstalled jax-0.7.2
Found existing installation: jaxlib 0.7.2
Uninstalling jaxlib-0.7.2:
  Successfully uninstalled jaxlib-0.7.2
Found existing installation: ml_dtypes 0.5.3
Uninstalling ml_dtypes-0.5.3:
  Successfully uninstalled ml_dtypes-0.5.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.4/71.4 MB 25.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 81.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 103.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.3/323.3 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.4/242.4 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.7/251.7 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 727.1/727.1 kB 27.1 MB/s eta 0:00:00
  Attempting un

In [ ]:
import os
os.kill(os.getpid(), 9)
print("done")

In [ ]:
!pip install "flwr[simulation]==1.9.0"

In [ ]:
pip uninstall -y numpy ml-dtypes jax jaxlib tensorflow

In [ ]:
pip install flwr[simulation]==1.9.0 numpy==1.26.4 ml-dtypes==0.5.0 protobuf==5.29.1

In [ ]:
!pip uninstall -y numpy ml-dtypes jax jaxlib tensorflow protobuf
!pip install flwr[simulation]==1.9.0 numpy==1.26.4 ml-dtypes==0.5.0 protobuf==4.25.3

In [ ]:
import flwr as fl
print(fl.__version__)

In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
import flwr
print("Flower version:", flwr.__version__)

In [ ]:
!pip uninstall -y flwr tensorflow ml-dtypes
!pip install flwr[simulation]==1.9.0

In [ ]:
import flwr
print(flwr.__version__)


In [ ]:
!flwr new @flwrlabs/quickstart-pytorch

In [ ]:
import flwr
print("Flower version:", flwr.__version__)

In [ ]:
!pip uninstall -y flwr tensorflow ml-dtypes
!pip install flwr[simulation]==1.9.0

In [ ]:
# ================================
# Suppress warnings and Ray logs
# ================================
import warnings, logging, os
warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.getLogger("ray").setLevel(logging.ERROR)

# ================================
# NumPy patch for Flower (Kaggle)
# ================================
import numpy as np
import numpy.lib.format as npformat

def safe_load(bytes_io, allow_pickle=False):
    bytes_io.seek(0)
    return npformat.read_array(bytes_io, allow_pickle=allow_pickle)

def safe_save(bytes_io, arr, allow_pickle=False):
    npformat.write_array(bytes_io, arr, allow_pickle=allow_pickle)

np.load = safe_load
np.save = safe_save

# ================================
# Imports
# ================================
import flwr as fl
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from collections import OrderedDict

# ================================
# Device
# ================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ================================
# Data preprocessing
# ================================
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

trainset = datasets.MNIST("./data", train=True, download=True, transform=transform)
testset  = datasets.MNIST("./data", train=False, download=True, transform=transform)

# ================================
# Non-IID loaders
# ================================
def get_non_iid_loader(digit_range):
    indices = [i for i, (_, label) in enumerate(trainset) if label in digit_range]
    subset = Subset(trainset, indices)
    return DataLoader(subset, batch_size=32, shuffle=True)

def get_test_loader():
    return DataLoader(testset, batch_size=32)

# ================================
# CNN Model
# ================================
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3)
        self.conv2 = nn.Conv2d(32, 64, 3)
        self.pool  = nn.MaxPool2d(2, 2)
        self.fc1   = nn.Linear(64 * 5 * 5, 128)
        self.fc2   = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(-1, 64 * 5 * 5)
        x = torch.relu(self.fc1(x))
        return torch.softmax(self.fc2(x), dim=1)

# ================================
# Flower Client
# ================================
class MnistClient(fl.client.NumPyClient):
    def __init__(self, cid):
        self.model = Net().to(device)
        self.optimizer = optim.Adam(self.model.parameters(), lr=0.001)
        self.loss_fn = nn.CrossEntropyLoss()
        self.cid = cid

        if cid == "0":
            self.trainloader = get_non_iid_loader(range(0, 4))
        elif cid == "1":
            self.trainloader = get_non_iid_loader(range(4, 7))
        else:
            self.trainloader = get_non_iid_loader(range(7, 10))

        self.testloader = get_test_loader()

    def get_parameters(self, config):
        return [v.cpu().numpy() for v in self.model.state_dict().values()]

    def set_parameters(self, parameters):
        state_dict = OrderedDict({
            k: torch.tensor(v) for k, v in zip(self.model.state_dict().keys(), parameters)
        })
        self.model.load_state_dict(state_dict, strict=True)

    def fit(self, parameters, config):
        self.set_parameters(parameters)
        self.model.train()

        for data, target in self.trainloader:
            data, target = data.to(device), target.to(device)
            self.optimizer.zero_grad()
            output = self.model(data)
            loss = self.loss_fn(output, target)
            loss.backward()
            self.optimizer.step()

        return self.get_parameters({}), len(self.trainloader.dataset), {}

    def evaluate(self, parameters, config):
        self.set_parameters(parameters)
        self.model.eval()
        correct, loss = 0, 0.0

        with torch.no_grad():
            for data, target in self.testloader:
                data, target = data.to(device), target.to(device)
                output = self.model(data)
                loss += self.loss_fn(output, target).item()
                pred = output.argmax(dim=1)
                correct += pred.eq(target).sum().item()

        accuracy = correct / len(self.testloader.dataset)
        return loss / len(self.testloader), len(self.testloader.dataset), {"accuracy": accuracy}

# ================================
# Strategy with Global Model Saving
# ================================
class SaveModelStrategy(fl.server.strategy.FedAvg):
    def __init__(self, num_rounds):
        super().__init__()
        self.num_rounds = num_rounds
        os.makedirs("models", exist_ok=True)

    def aggregate_fit(self, rnd, results, failures):
        aggregated_parameters, aggregated_metrics = super().aggregate_fit(
            rnd, results, failures
        )

        if aggregated_parameters is not None and rnd == self.num_rounds:
            print("\n💾 Saving final global model...")

            model = Net()
            params_dict = zip(
                model.state_dict().keys(),
                fl.common.parameters_to_ndarrays(aggregated_parameters)
            )

            state_dict = OrderedDict(
                {k: torch.tensor(v) for k, v in params_dict}
            )

            model.load_state_dict(state_dict)
            torch.save(model.state_dict(), "models/global_model.pth")

            print("✅ Global model saved at models/global_model.pth\n")

        return aggregated_parameters, aggregated_metrics

# ================================
# Run Federated Simulation
# ================================
NUM_ROUNDS = 3

fl.simulation.start_simulation(
    client_fn=lambda cid: MnistClient(cid).to_client(),
    num_clients=3,
    config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
    strategy=SaveModelStrategy(num_rounds=NUM_ROUNDS),
)


In [ ]:
!pip uninstall -y numpy
!pip install numpy==1.23.5

In [ ]:
pip install --upgrade flwr[simulation] numpy==1.26.4 ml-dtypes==0.5.0 protobuf==4.25.3

In [ ]:
import flwr as fl
print("Flower version:", fl.__version__)

In [ ]:
!pip uninstall -y flwr
!pip install --no-cache-dir flwr[simulation]==1.23.0

In [ ]:
import flwr as fl
print("Flower version:", fl.__version__)

In [1]:
!pip uninstall -y flwr
!pip cache purge

Found existing installation: flwr 1.25.0
Uninstalling flwr-1.25.0:
  Successfully uninstalled flwr-1.25.0
Files removed: 88


In [ ]:
pip install --no-cache-dir flwr[simulation]==1.24.0


In [2]:
!pip uninstall -y flwr
!pip install --no-cache-dir flwr[simulation]==1.24.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 787.9/787.9 kB 107.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.3/323.3 kB 304.1 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.1
    Uninstalling protobuf-5.29.1:
      Successfully uninstalled protobuf-5.29.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.22.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
orbax-checkpoint 0.11.25 requires jax>=0.6.0, which is not installed.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
flax 0.10.7 requires jax>=0.6.0, which is not installed.
google-cloud-translate 3.12.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.19.5, but you have protobuf 6.33.4 which is incompatibl

In [3]:
import flwr as fl
print("Flower version:", fl.__version__)

Flower version: 1.24.0


In [ ]:
 #================================
# Suppress warnings and Ray logs
# ================================
import warnings, logging, os
warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.getLogger("ray").setLevel(logging.ERROR)

# ================================
# NumPy patch for Flower (Kaggle)
# ================================
import numpy as np
import numpy.lib.format as npformat

def safe_load(bytes_io, allow_pickle=False):
    bytes_io.seek(0)
    return npformat.read_array(bytes_io, allow_pickle=allow_pickle)

def safe_save(bytes_io, arr, allow_pickle=False):
    npformat.write_array(bytes_io, arr, allow_pickle=allow_pickle)

np.load = safe_load
np.save = safe_save

# ================================
# Imports
# ================================
import flwr as fl
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from collections import OrderedDict

# ================================
# Device
# ================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ================================
# Data preprocessing
# ================================
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

trainset = datasets.MNIST("./data", train=True, download=True, transform=transform)
testset  = datasets.MNIST("./data", train=False, download=True, transform=transform)

# ================================
# Non-IID loaders
# ================================
def get_non_iid_loader(digit_range):
    indices = [i for i, (_, label) in enumerate(trainset) if label in digit_range]
    subset = Subset(trainset, indices)
    return DataLoader(subset, batch_size=32, shuffle=True)

def get_test_loader():
    return DataLoader(testset, batch_size=32)

# ================================
# CNN Model
# ================================
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3)
        self.conv2 = nn.Conv2d(32, 64, 3)
        self.pool  = nn.MaxPool2d(2, 2)
        self.fc1   = nn.Linear(64 * 5 * 5, 128)
        self.fc2   = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(-1, 64 * 5 * 5)
        x = torch.relu(self.fc1(x))
        return torch.softmax(self.fc2(x), dim=1)

# ================================
# Flower Client
# ================================
class MnistClient(fl.client.NumPyClient):
    def __init__(self, cid):
        self.model = Net().to(device)
        self.optimizer = optim.Adam(self.model.parameters(), lr=0.001)
        self.loss_fn = nn.CrossEntropyLoss()
        self.cid = cid

        if cid == "0":
            self.trainloader = get_non_iid_loader(range(0, 4))
        elif cid == "1":
            self.trainloader = get_non_iid_loader(range(4, 7))
        else:
            self.trainloader = get_non_iid_loader(range(7, 10))

        self.testloader = get_test_loader()

    def get_parameters(self, config):
        return [v.cpu().numpy() for v in self.model.state_dict().values()]

    def set_parameters(self, parameters):
        state_dict = OrderedDict({
            k: torch.tensor(v) for k, v in zip(self.model.state_dict().keys(), parameters)
        })
        self.model.load_state_dict(state_dict, strict=True)

    def fit(self, parameters, config):
        self.set_parameters(parameters)
        self.model.train()

        for data, target in self.trainloader:
            data, target = data.to(device), target.to(device)
            self.optimizer.zero_grad()
            output = self.model(data)
            loss = self.loss_fn(output, target)
            loss.backward()
            self.optimizer.step()

        return self.get_parameters({}), len(self.trainloader.dataset), {}

    def evaluate(self, parameters, config):
        self.set_parameters(parameters)
        self.model.eval()
        correct, loss = 0, 0.0

        with torch.no_grad():
            for data, target in self.testloader:
                data, target = data.to(device), target.to(device)
                output = self.model(data)
                loss += self.loss_fn(output, target).item()
                pred = output.argmax(dim=1)
                correct += pred.eq(target).sum().item()

        accuracy = correct / len(self.testloader.dataset)
        return loss / len(self.testloader), len(self.testloader.dataset), {"accuracy": accuracy}

# ================================
# Strategy with Global Model Saving
# ================================
class SaveModelStrategy(fl.server.strategy.FedAvg):
    def __init__(self, num_rounds):
        super().__init__()
        self.num_rounds = num_rounds
        os.makedirs("models", exist_ok=True)

    def aggregate_fit(self, rnd, results, failures):
        aggregated_parameters, aggregated_metrics = super().aggregate_fit(
            rnd, results, failures
        )

        if aggregated_parameters is not None and rnd == self.num_rounds:
            print("\n💾 Saving final global model...")

            model = Net()
            params_dict = zip(
                model.state_dict().keys(),
                fl.common.parameters_to_ndarrays(aggregated_parameters)
            )

            state_dict = OrderedDict(
                {k: torch.tensor(v) for k, v in params_dict}
            )

            model.load_state_dict(state_dict)
            torch.save(model.state_dict(), "models/global_model.pth")

            print("✅ Global model saved at models/global_model.pth\n")

        return aggregated_parameters, aggregated_metrics

# ================================
# Run Federated Simulation
# ================================
NUM_ROUNDS = 3

fl.simulation.start_simulation(
    client_fn=lambda cid: MnistClient(cid).to_client(),
    num_clients=3,
    config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
    strategy=SaveModelStrategy(num_rounds=NUM_ROUNDS),
)

In [ ]:
# List files in the models directory
!ls -lh models/

In [ ]:
import torch

# Recreate your model architecture
model = Net()   # <-- use the same Net class you defined earlier

# Load the saved weights
model.load_state_dict(torch.load("models/global_model.pth"))

print("✅ Model reloaded successfully")
print("Model structure:\n", model)

# Optional: peek at the first few keys in the state_dict
print("Saved parameters:", list(model.state_dict().keys())[:5])

In [ ]:
import torch

# Recreate your model architecture (same Net class you used in training)
model = Net()

# Load the saved weights
model.load_state_dict(torch.load("models/global_model.pth"))

# Print confirmation
print("✅ Global model reloaded successfully")

# Print the model structure
print(model)

# Print the names of parameters saved
print("\n🔎 Saved parameters:")
for name, param in model.state_dict().items():
    print(f"{name}: {param.shape}")

In [ ]:
# ================================
# Suppress warnings and Ray logs
# ================================
import warnings, logging, os
warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.getLogger("ray").setLevel(logging.ERROR)

# ================================
# NumPy patch for Flower (Kaggle)
# ================================
import numpy as np
import numpy.lib.format as npformat

def safe_load(bytes_io, allow_pickle=False):
    bytes_io.seek(0)
    return npformat.read_array(bytes_io, allow_pickle=allow_pickle)

def safe_save(bytes_io, arr, allow_pickle=False):
    npformat.write_array(bytes_io, arr, allow_pickle=allow_pickle)

np.load = safe_load
np.save = safe_save

# ================================
# Imports
# ================================
import flwr as fl
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from collections import OrderedDict
import json

# ================================
# Device
# ================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ================================
# Data preprocessing
# ================================
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

trainset = datasets.MNIST("./data", train=True, download=True, transform=transform)
testset  = datasets.MNIST("./data", train=False, download=True, transform=transform)

# ================================
# Non-IID loaders
# ================================
def get_non_iid_loader(digit_range):
    indices = [i for i, (_, label) in enumerate(trainset) if label in digit_range]
    subset = Subset(trainset, indices)
    return DataLoader(subset, batch_size=32, shuffle=True)

def get_test_loader():
    return DataLoader(testset, batch_size=32)

# ================================
# CNN Model
# ================================
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3)
        self.conv2 = nn.Conv2d(32, 64, 3)
        self.pool  = nn.MaxPool2d(2, 2)
        self.fc1   = nn.Linear(64 * 5 * 5, 128)
        self.fc2   = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(-1, 64 * 5 * 5)
        x = torch.relu(self.fc1(x))
        return torch.softmax(self.fc2(x), dim=1)

# ================================
# Flower Client
# ================================
class MnistClient(fl.client.NumPyClient):
    def __init__(self, cid):
        self.model = Net().to(device)
        self.optimizer = optim.Adam(self.model.parameters(), lr=0.001)
        self.loss_fn = nn.CrossEntropyLoss()
        self.cid = cid

        if cid == "0":
            self.trainloader = get_non_iid_loader(range(0, 4))
        elif cid == "1":
            self.trainloader = get_non_iid_loader(range(4, 7))
        else:
            self.trainloader = get_non_iid_loader(range(7, 10))

        self.testloader = get_test_loader()

    def get_parameters(self, config):
        return [v.cpu().numpy() for v in self.model.state_dict().values()]

    def set_parameters(self, parameters):
        state_dict = OrderedDict({
            k: torch.tensor(v) for k, v in zip(self.model.state_dict().keys(), parameters)
        })
        self.model.load_state_dict(state_dict, strict=True)

    def fit(self, parameters, config):
        self.set_parameters(parameters)
        self.model.train()

        for data, target in self.trainloader:
            data, target = data.to(device), target.to(device)
            self.optimizer.zero_grad()
            output = self.model(data)
            loss = self.loss_fn(output, target)
            loss.backward()
            self.optimizer.step()

        return self.get_parameters({}), len(self.trainloader.dataset), {}

    def evaluate(self, parameters, config):
        self.set_parameters(parameters)
        self.model.eval()
        correct, loss = 0, 0.0

        with torch.no_grad():
            for data, target in self.testloader:
                data, target = data.to(device), target.to(device)
                output = self.model(data)
                loss += self.loss_fn(output, target).item()
                pred = output.argmax(dim=1)
                correct += pred.eq(target).sum().item()

        accuracy = correct / len(self.testloader.dataset)
        return loss / len(self.testloader), len(self.testloader.dataset), {"accuracy": accuracy}

# ================================
# Strategy with Global Model + Metrics Saving
# ================================
class SaveModelStrategy(fl.server.strategy.FedAvg):
    def __init__(self, num_rounds):
        super().__init__()
        self.num_rounds = num_rounds
        os.makedirs("models", exist_ok=True)
        self.history = []  # track loss/accuracy per round

    def aggregate_evaluate(self, rnd, results, failures):
        aggregated_loss, aggregated_metrics = super().aggregate_evaluate(rnd, results, failures)

        if aggregated_metrics is not None:
            self.history.append({
                "round": rnd,
                "loss": aggregated_loss,
                "metrics": aggregated_metrics
            })
            print(f"📊 Round {rnd} - Loss: {aggregated_loss:.4f}, Metrics: {aggregated_metrics}")

        return aggregated_loss, aggregated_metrics

    def aggregate_fit(self, rnd, results, failures):
        aggregated_parameters, aggregated_metrics = super().aggregate_fit(rnd, results, failures)

        if aggregated_parameters is not None and rnd == self.num_rounds:
            print("\n💾 Saving final global model and metrics...")

            # Save model weights
            model = Net()
            params_dict = zip(
                model.state_dict().keys(),
                fl.common.parameters_to_ndarrays(aggregated_parameters)
            )
            state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
            model.load_state_dict(state_dict)
            torch.save(model.state_dict(), "models/global_model.pth")

            # Save metrics history
            with open("models/training_history.json", "w") as f:
                json.dump(self.history, f, indent=4)

            print("✅ Global model saved at models/global_model.pth")
            print("✅ Training history saved at models/training_history.json\n")

        return aggregated_parameters, aggregated_metrics

# ================================
# Run Federated Simulation
# ================================
NUM_ROUNDS = 3

fl.simulation.start_simulation(
    client_fn=lambda cid: MnistClient(cid).to_client(),
    num_clients=3,
    config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
    strategy=SaveModelStrategy(num_rounds=NUM_ROUNDS),
)

In [ ]:
!ls -lh models/

In [ ]:
import json

with open("models/training_history.json") as f:
    history = json.load(f)

for entry in history:
    loss = entry.get("loss")
    acc = entry["metrics"].get("accuracy")

    # Handle None values gracefully
    loss_str = f"{loss:.4f}" if loss is not None else "N/A"
    acc_str = f"{acc:.4f}" if acc is not None else "N/A"

    print(f"Round {entry['round']}: Loss={loss_str}, Accuracy={acc_str}")

In [4]:
# ================================
# Suppress warnings and Ray logs
# ================================
import warnings, logging, os
warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.getLogger("ray").setLevel(logging.ERROR)

# ================================
# NumPy patch for Flower (Kaggle)
# ================================
import numpy as np
import numpy.lib.format as npformat

def safe_load(bytes_io, allow_pickle=False):
    bytes_io.seek(0)
    return npformat.read_array(bytes_io, allow_pickle=allow_pickle)

def safe_save(bytes_io, arr, allow_pickle=False):
    npformat.write_array(bytes_io, arr, allow_pickle=allow_pickle)

np.load = safe_load
np.save = safe_save

# ================================
# Imports
# ================================
import flwr as fl
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from collections import OrderedDict
import json

# ================================
# Device
# ================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ================================
# Data preprocessing
# ================================
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

trainset = datasets.MNIST("./data", train=True, download=True, transform=transform)
testset  = datasets.MNIST("./data", train=False, download=True, transform=transform)

# ================================
# Metric aggregation function
# ================================
def weighted_average(metrics):
    accuracies = []
    examples = []

    for num_examples, m in metrics:
        if "accuracy" in m:
            accuracies.append(m["accuracy"] * num_examples)
            examples.append(num_examples)

    return {"accuracy": sum(accuracies) / sum(examples)}


# ================================
# Non-IID loaders
# ================================
def get_non_iid_loader(digit_range):
    indices = [i for i, (_, label) in enumerate(trainset) if label in digit_range]
    subset = Subset(trainset, indices)
    return DataLoader(subset, batch_size=32, shuffle=True)

def get_test_loader():
    return DataLoader(testset, batch_size=32)

# ================================
# CNN Model
# ================================
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3)
        self.conv2 = nn.Conv2d(32, 64, 3)
        self.pool  = nn.MaxPool2d(2, 2)
        self.fc1   = nn.Linear(64 * 5 * 5, 128)
        self.fc2   = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(-1, 64 * 5 * 5)
        x = torch.relu(self.fc1(x))
        return torch.softmax(self.fc2(x), dim=1)

# ================================
# Flower Client
# ================================
class MnistClient(fl.client.NumPyClient):
    def __init__(self, cid):
        self.model = Net().to(device)
        self.optimizer = optim.Adam(self.model.parameters(), lr=0.001)
        self.loss_fn = nn.CrossEntropyLoss()
        self.cid = cid

        if cid == "0":
            self.trainloader = get_non_iid_loader(range(0, 4))
        elif cid == "1":
            self.trainloader = get_non_iid_loader(range(4, 7))
        else:
            self.trainloader = get_non_iid_loader(range(7, 10))

        self.testloader = get_test_loader()

    def get_parameters(self, config):
        return [v.cpu().numpy() for v in self.model.state_dict().values()]

    def set_parameters(self, parameters):
        state_dict = OrderedDict({
            k: torch.tensor(v) for k, v in zip(self.model.state_dict().keys(), parameters)
        })
        self.model.load_state_dict(state_dict, strict=True)

    def fit(self, parameters, config):
        self.set_parameters(parameters)
        self.model.train()

        for data, target in self.trainloader:
            data, target = data.to(device), target.to(device)
            self.optimizer.zero_grad()
            output = self.model(data)
            loss = self.loss_fn(output, target)
            loss.backward()
            self.optimizer.step()

        return self.get_parameters({}), len(self.trainloader.dataset), {}

    def evaluate(self, parameters, config):
        self.set_parameters(parameters)
        self.model.eval()
        correct, loss = 0, 0.0

        with torch.no_grad():
            for data, target in self.testloader:
                data, target = data.to(device), target.to(device)
                output = self.model(data)
                loss += self.loss_fn(output, target).item()
                pred = output.argmax(dim=1)
                correct += pred.eq(target).sum().item()

        accuracy = correct / len(self.testloader.dataset)
        return loss / len(self.testloader), len(self.testloader.dataset), {"accuracy": accuracy}

# ================================
# Strategy with Global Model + Metrics Saving
# ================================
class SaveModelStrategy(fl.server.strategy.FedAvg):
    def __init__(self, num_rounds):
        super().__init__(
            evaluate_metrics_aggregation_fn=weighted_average
        )
        self.num_rounds = num_rounds
        os.makedirs("models", exist_ok=True)
        self.history = []  # track loss/accuracy per round

    def aggregate_evaluate(self, rnd, results, failures):
        aggregated_loss, aggregated_metrics = super().aggregate_evaluate(rnd, results, failures)

        accuracy = None
        if aggregated_metrics and "accuracy" in aggregated_metrics:
            accuracy = aggregated_metrics["accuracy"]
    
        self.history.append({
            "round": rnd,
            "loss": aggregated_loss,
            "accuracy": accuracy
        })
    
        print(f"📊 Round {rnd} - Loss: {aggregated_loss:.4f}, Accuracy: {accuracy:.4f}")
    
        return aggregated_loss, aggregated_metrics


    def aggregate_fit(self, rnd, results, failures):
        aggregated_parameters, aggregated_metrics = super().aggregate_fit(rnd, results, failures)

        if aggregated_parameters is not None and rnd == self.num_rounds:
            print("\n💾 Saving final global model and metrics...")

            # Save model weights
            model = Net()
            params_dict = zip(
                model.state_dict().keys(),
                fl.common.parameters_to_ndarrays(aggregated_parameters)
            )
            state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
            model.load_state_dict(state_dict)
            torch.save(model.state_dict(), "models/global_model.pth")

            # Save metrics history
            with open("models/training_history.json", "w") as f:
                json.dump(self.history, f, indent=4)

            print("✅ Global model saved at models/global_model.pth")
            print("✅ Training history saved at models/training_history.json\n")

        return aggregated_parameters, aggregated_metrics

# ================================
# Run Federated Simulation
# ================================
NUM_ROUNDS = 11

fl.simulation.start_simulation(
    client_fn=lambda cid: MnistClient(cid).to_client(),
    num_clients=3,
    config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
    strategy=SaveModelStrategy(num_rounds=NUM_ROUNDS),
)

Using device: cpu


100%|██████████| 9.91M/9.91M [00:00<00:00, 135MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 23.0MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 133MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.45MB/s]
2026-01-25 11:08:52.417204: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769339332.626349     136 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769339332.686674     136 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769339333.168447     136 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769339333.168500     136 computation_pl

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...


💾 Saving final global model and metrics...
✅ Global model saved at models/global_model.pth
✅ Training history saved at models/training_history.json



(ClientAppActor pid=311) 
(ClientAppActor pid=311)         
(ClientAppActor pid=311) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid">}. You can import the `Context` like this: `from flwr.common import Context` [repeated 3x across cluster]
(ClientAppActor pid=311)             This is a deprecated feature. It will be removed [repeated 3x across cluster]
(ClientAppActor pid=311)             entirely in future versions of Flower. [repeated 3x across cluster]
(ClientAppActor pid=310) 
(ClientAppActor pid=310)         
(ClientAppActor pid=313) 
(ClientAppActor pid=313)         
INFO :      aggregate_evaluate: received 3 results and 0 failures
INFO :      
INFO :      [SUMMARY]
INFO :      Run finished 11 round(s) in 922.11s
INFO :      	History (loss, distributed):
INFO :      		round 1: 1.9670725501002593
INFO :      		round 2: 1.8513214969025633
INFO :      		round 3: 1.70

📊 Round 11 - Loss: 1.6303, Accuracy: 0.8315


History (loss, distributed):
	round 1: 1.9670725501002593
	round 2: 1.8513214969025633
	round 3: 1.702354839434639
	round 4: 1.7149118520200446
	round 5: 1.6047380423774353
	round 6: 1.6304494310110902
	round 7: 1.59835498211102
	round 8: 1.622092625203605
	round 9: 1.6351078108857613
	round 10: 1.562934866347633
	round 11: 1.6302657542518153
History (metrics, distributed, evaluate):
{'accuracy': [(1, 0.5627),
              (2, 0.6302),
              (3, 0.784),
              (4, 0.7504999999999998),
              (5, 0.8699),
              (6, 0.8356),
              (7, 0.8675),
              (8, 0.8394),
              (9, 0.826),
              (10, 0.9005),
              (11, 0.8315)]}

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
